# Problem Statement

## From Researcher
This dataset gathered SSVEP-BCI recordings of 35 healthy subjects (17 females, aged 17-34 years, mean age: 22 years) focusing on 40 characters flickering at different frequencies (8-15.8 Hz with an interval of 0.2 Hz). For each subject, the experiment consisted of 6 blocks. Each block contained 40 trials corresponding to all 40 characters indicated in a random order. Each trial started with a visual cue (a red square) indicating a target stimulus. The cue appeared for 0.5 s on the screen. Subjects were asked to shift their gaze to the target as soon as possible within the cue duration. Following the cue offset, all stimuli started to flicker on the screen concurrently and lasted 5 s. After stimulus offset, the screen was blank for 0.5 s before the next trial began, which allowed the subjects to have short breaks between consecutive trials. Each trial lasted a total of 6 s. To facilitate visual fixation, a red triangle appeared below the flickering target during the stimulation period. In each block, subjects were asked to avoid eye blinks during the stimulation period. To avoid visual fatigue, there was a rest for several minutes between two consecutive blocks.

EEG data were acquired using a Synamps2 system (Neuroscan, Inc.) with a sampling rate of 1000 Hz. The amplifier frequency passband ranged from 0.15 Hz to 200 Hz. Sixty-four channels covered the whole scalp of the subject and were aligned according to the international 10-20 system. The ground was placed on midway between Fz and FPz. The reference was located on the vertex. Electrode impedances were kept below 10 K¶∏. To remove the common power-line noise, a notch filter at 50 Hz was applied in data recording. Event triggers generated by the computer to the amplifier and recorded on an event channel synchronized to the EEG data. 

The continuous EEG data was segmented into 6 s epochs (500 ms pre-stimulus, 5.5 s post-stimulus onset). The epochs were subsequently downsampled to 250 Hz. Thus each trial consisted of 1500 time points. Finally, these data were stored as double-precision floating-point values in MATLAB and were named as subject indices (i.e., S01.mat, °≠, S35.mat). For each file, the data loaded in MATLAB generate a 4-D matrix named °Ædata°Ø with dimensions of [64, 1500, 40, 6]. The four dimensions indicate °ÆElectrode index°Ø, °ÆTime points°Ø, °ÆTarget index°Ø, and °ÆBlock index°Ø. The electrode positions were saved in a °Æ64-channels.loc°Ø file. Six trials were available for each SSVEP frequency. Frequency and phase values for the 40 target indices were saved in a °ÆFreq_Phase.mat°Ø file.

Information for all subjects was listed in a °ÆSub_info.txt°Ø file. For each subject, there are five factors including °ÆSubject Index°Ø, °ÆGender°Ø, °ÆAge°Ø, °ÆHandedness°Ø, and °ÆGroup°Ø. Subjects were divided into an °Æexperienced°Ø group (eight subjects, S01-S08) and a °Ænaive°Ø group (27 subjects, S09-S35) according to their experience in SSVEP-based BCIs.


## Summary
1.วัดหัว 64 จุด sampling rate 250 Hz

2.มีคน 35 คน แต่ละคนทำการทดลอง 6 blocks มีเวลาพัก 2-3 mins ระหว่าง block ที่ติดกันครับ

3.ใน 1 Block ประกอบ 40 trials (40 frequencies & phases as presented in an image, random sequence) แต่ละ trial สัญญาณยาว 6 วินาที

4.ใน 1 trial ประกอบไปด้วย 0.5s สำหรับ cue ว่าให้มองที่ไหนจาก 1ใน 40 เป้าหมาย, 5s การ stimulation สมองด้วยความถี่และเฟสนั้นๆ, หลังจากนั้น0.5s จอว่าง แล้วก็วนกลับไป random เป้าหมายใหม่จนครบ 40 ดังข้อ3

## Reference
1. Main Research Paper
    - [A Benchmark Dataset for SSVEP-Based Brain-Computer Interfaces](https://www.researchgate.net/publication/309897862_A_Benchmark_Dataset_for_SSVEP-Based_Brain-Computer_Interfaces)
2. CCA Implementation:
    - [Combining the Benefits of CCA and SVMs for SSVEP-based BCIs in Real-world Conditions](https://www.researchgate.net/publication/320572057_Combining_the_Benefits_of_CCA_and_SVMs_for_SSVEP-based_BCIs_in_Real-world_Conditions/download)
    - https://stats.stackexchange.com/questions/77287/canonical-correlation-analysis-without-raw-data-algebra-of-cca/77309#77309
    - https://stats.stackexchange.com/questions/65692/how-to-visualize-what-canonical-correlation-analysis-does-in-comparison-to-what
3. How to solve for Maximum Canonical Correlation
    - [A Tutorial on Canonical Correlation Methods](https://arxiv.org/abs/1711.02391)
4. Best Electrodes to use
    - [An online multi-channel SSVEP-based brain-computer interface using a canonical correlation analysis method.](https://www.ncbi.nlm.nih.gov/pubmed/19494422)

In [1]:
%matplotlib inline

In [2]:
%%javascript
// Disable scroll bar
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

<IPython.core.display.Javascript object>

In [2]:
!pip install xarray

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   - -------------------------------------- 0.0/1.2 MB 991.0 kB/s eta 0:00:02
   ---------- ----------------------------- 0.3/1.2 MB 4.1 MB/s eta 0:00:01
   --------------------- ------------------ 0.7/1.2 MB 5.4 MB/s eta 0:00:01
   -------------------------------- ------- 1.0/1.2 MB 5.8 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 6.1 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Imports

In [82]:
import scipy.io
from scipy.signal import butter, lfilter
import pandas as pd
import xarray as xr
import numpy as np
from pprint import pprint
from functools import reduce
import re
from matplotlib import pyplot as plt
from IPython.display import display, Markdown

# Parameters

In [57]:
# Parameters for Benchmark dataset
fs = 250
n_targets = 10
t_cue = 0.5
n_trials = 6
t_select = 1

# Parameters for Butterworth filter
lowcut = 7
highcut = 40

# Parater for CCA classification
n_harmonics = 6
PI = np.pi

# an index of time frame after 0.5s * 250Hz has passed
post_stimulus_time_index = int(t_cue * fs) # SSVEP 开始前的提示信号

# Functions

In [58]:
# 从多维数组中提取当前频率、当前次实验中每个电极的数据
def get_input_data(raw_data, electrodes, target_id, block_id):
    interest_electrode_ids = []
    for electrode in electrodes:
        interest_electrode_ids.append(elec_to_elec_id_map[electrode])
    result = {}
    for electrode_id in interest_electrode_ids:
        values = []
        for t in range(len(raw_data[electrode_id])):
            values.append(raw_data[electrode_id][t][target_id][block_id])
        result[elec_id_to_elec_map[electrode_id]] = values
    return result

In [59]:
def bandpass_filter(data, lowcut, highcut, fs, order=5):
    nyq  = 0.5 * fs  # Nyquist Frequency
    low  = lowcut / nyq
    high = highcut /nyq
    b, a = butter(order, [low, high], btype='band')
    y    = lfilter(b, a, data)


In [60]:
# 所有谐波相加得到参考信号中 t 时刻的信号采样点
def generate_reference_signal_at_time(f, t, max_harmonic, phase):
    values = []
    for h in range(1, max_harmonic + 1):
        values += ref_wave(f, h, t, phase)
    return values

def generate_reference_signal(frequency, sampling_frequency, total_time, max_harmonic, phase):
    ref_signal = []
    num_time_step = int(total_time * sampling_frequency)
    for step in range(num_time_step):
        time = step * 1/sampling_frequency
        ref_signal_at_t = generate_reference_signal_at_time(frequency, time, max_harmonic, phase)
        ref_signal.append(ref_signal_at_t)
    return ref_signal

In [61]:
def find_maximum_canonical_correlations(X, Y):
    if X.shape[0] == Y.shape[0]:
        N = X.shape[0]
    else:
        print('time frame is not equal')
        return None
    C_xx = 1/N * (X.T @ X) # X 自相关
    C_yy = 1/N * (Y.T @ Y) # Y 自相关
    C_xy = 1/N * (X.T @ Y) # X, Y 互相关
    C_yx = 1/N * (Y.T @ X)
    C_xx_inv = np.linalg.pinv(C_xx) # X 自相关的逆矩阵
    C_yy_inv = np.linalg.pinv(C_yy)
    eig_values, eig_vectors = scipy.linalg.eig(C_yy_inv @ C_yx @ C_xx_inv @ C_xy)
    sqrt_eig_values = np.sqrt(eig_values)
    return max(sqrt_eig_values)

# Load data

In [62]:
# Map electrode names to thier index for easier read/use
f = open("D:/2024/7-9/dissertation/code/64-channels.loc", "r")
lines = re.split('\n', f.read())
electrodes = [] #二维列表
for line in lines:
    electrodes.append(list(filter(lambda text: len(text) > 0, re.split('\s', line)))) # 去除每行分割后的空元素
electrodes = list(filter(lambda l: len(l) > 0, electrodes)) #去除空白的行

#建立电极名称和电极ID的相互映射
elec_to_elec_id_map = {}
elec_id_to_elec_map = {}
for i in range(len(electrodes)):
    elec_to_elec_id_map[electrodes[i][3]] = int(electrodes[i][0])
    elec_id_to_elec_map[int(electrodes[i][0])] = electrodes[i][3]

<>:6: SyntaxWarning: invalid escape sequence '\s'
<>:6: SyntaxWarning: invalid escape sequence '\s'
C:\Users\qianqian\AppData\Local\Temp\ipykernel_3672\3885769207.py:6: SyntaxWarning: invalid escape sequence '\s'
  electrodes.append(list(filter(lambda text: len(text) > 0, re.split('\s', line)))) # 去除每行分割后的空元素


In [63]:
# from ftp://sccn.ucsd.edu/pub/ssvep_benchmark_dataset/
main_data_file = 'D:/2024/7-9/dissertation/code/S1.mat' #仅研究第一个受试者

In [64]:
mat = scipy.io.loadmat(main_data_file)
raw_data = mat['data'] # (channels, samples, targets, trials)

In [65]:
# print(raw_data.shape[1])

In [66]:
target_setting_info = scipy.io.loadmat('D:/2024/7-9/dissertation/code/Freq_Phase.mat')

In [67]:
# 2列数据，1列存储频率，1列存储相位
target_setting = pd.DataFrame.from_dict({
    "frequency": target_setting_info['freqs'][0], # 8, 9, ..., 15, 8.2, 9.2, ..., 15.2, ..., 8.8, 9.8, ..., 15.8 Hz
    "phase": target_setting_info['phases'][0]
})

## Reformat SSVEP data into DataFrame

In [68]:
# Chosen from Reference Paper #4
interest_channels = ['P3', 'PZ', 'P4', 'PO7', 'POz', 'O1', 'Oz', 'O2', 'PO8']
# 10 out of 40 
target_ids = range(n_targets) #选择前 10 个频率进行分类 (8, 9, ..., 15, 8.2, 9.2 Hz) -- (0, 1, ..., 9)
trial_ids = range(n_trials) # trials

In [91]:
X = {} # 最终结构 (target, trial, channel, sample)
for target_id in target_ids:
    if target_id not in X:
        X[target_id] = {} #如果 target_id 不存在于字典 X 中，为其初始化一个空字典
    for trial_id in trial_ids:
        if trial_id not in X[target_id]:
            X[target_id][trial_id] = {}
        
        X[target_id][trial_id] = pd.DataFrame.from_dict(
            get_input_data(raw_data, interest_channels, target_id, trial_id)
        )[post_stimulus_time_index:post_stimulus_time_index+t_select*fs] # 数据段从提示信号结束之后开始
        



ValueError: selected axis is out of range

In [89]:
 X[target_ids[0]][trial_ids[0]].head() # 返回第一个频率，第一次实验数据中所有通道的前 5 个采样值
# X[target_ids[0]][trial_ids[0]].shape

AttributeError: 'numpy.ndarray' object has no attribute 'head'

In [29]:
'''
print('channel:', interest_channels[0])
print('number of data points:', len(X[target_ids[0]][trial_ids[0]][interest_channels[0]]))
plt.plot(X[target_ids[0]][trial_ids[0]][interest_channels[0]])
'''

"\nprint('channel:', interest_channels[0])\nprint('number of data points:', len(X[target_ids[0]][trial_ids[0]][interest_channels[0]]))\nplt.plot(X[target_ids[0]][trial_ids[0]][interest_channels[0]])\n"

## Generate Reference Signal

In [36]:
sin = lambda f, h, t, p: np.sin(2*PI*f*h*t + p) # 定义正弦函数
cos = lambda f, h, t, p: np.cos(2*PI*f*h*t + p)
ref_wave = lambda f, h, t, p: [sin(f, h, t, p), cos(f, h, t, p)]

In [41]:
Y = {}
for setting_index in target_setting.index:
    frequency, phase = target_setting.iloc[setting_index] # 提取每个频率和相位
    signal = generate_reference_signal(
        frequency = frequency,
        sampling_frequency = fs,
        total_time = t_cue + t_select,
        max_harmonic = n_harmonics,
        phase = phase
    )
    Y[f'setting_{setting_index}'] = pd.DataFrame(signal[post_stimulus_time_index:])

In [42]:
# columns are [sin, cos] * [number of harmonic] (2 * 6 = 12)
# rows are time steps
# Example Reference Signal
# Y['setting_0'].head()
# Y['setting_0'].shape

In [43]:
# plt.plot(Y['setting_0'][0])

## Optimization to find max ρ (canonial correlation)

In [48]:
result = {}
# X[target_id][block_id]
for target_id in target_ids:
    if target_id not in result:
        result[target_id] = {}
    for trial_id in trial_ids:
        if trial_id not in result[target_id]:
            result[target_id][trial_id] = {}
        keys = list(Y.keys()) # [cos, sin] * [harmonics] (0 - 11)
        for ref_signal_id in keys:
            value = find_maximum_canonical_correlations(
                X = X[target_id][trial_id].values,
                Y = Y[ref_signal_id].values
            )
            if value.imag == 0.0:
                result[target_id][trial_id][ref_signal_id] = value.real
            else:
                result[target_id][trial_id][ref_signal_id] = None

## Classification & Accuracy of the algorithm

In [52]:
for target_id in target_ids:
    display(Markdown(f"## Target: {target_id}"))
    df = pd.DataFrame.from_dict(result[target_id])
    display(Markdown('### Classification:'))
    num_correct= 0
    for trial_id in trial_ids:
        _, top_correlation_setting = df[trial_id].idxmax().split('_')
        is_correct = int(top_correlation_setting) == target_id
        if is_correct:
            num_correct += 1
        display(
            Markdown(f'Trial #{trial_id} Correct output should be {target_id}, classify as {top_correlation_setting}: **{is_correct}**')
        )
    display(
        Markdown('Accuracy: **%.2f%%**'%(num_correct/len(trial_ids)*100))
    )
    display(Markdown('### Raw Max CC Value (rows are blocks)'))
    display(df)

## Target: 0

### Classification:

Block #0 Correct output should be 0, classify as 0: **True**

Block #1 Correct output should be 0, classify as 26: **False**

Block #2 Correct output should be 0, classify as 21: **False**

...

Block #4 Correct output should be 9, classify as 9: **True**

Block #5 Correct output should be 9, classify as 17: **False**

Accuracy: **50.00%**

### Raw Max CC Value (rows are blocks)

,0,1,2,3,4,5
setting_0,0.483533,0.387034,0.478812,0.509068,0.506986,0.434487
setting_1,0.541865,0.555897,0.615637,0.500669,0.502711,0.591168
setting_2,0.553398,0.346822,0.333846,0.441853,0.411925,0.590370
setting_3,0.511009,0.391779,0.539201,0.505087,0.411966,0.438154
setting_4,0.382999,0.520057,0.424771,0.374163,0.420205,0.493575
setting_5,0.552569,0.469166,0.358569,0.363184,0.389376,0.482415
setting_6,0.303037,0.473482,0.360401,0.327833,0.321570,0.419137
setting_7,0.356513,0.325198,0.379958,0.335007,0.338851,0.303418
setting_8,0.462054,0.455893,0.489721,0.493116,0.422535,0.402710
setting_9,0.646287,0.613108,0.552173,0.504585,0.524427,0.646262
